# Day 21: Multi-GPU Instances & Multi-Instance GPU (MIG)
> *100 Days of Inference* | Layer: **Infrastructure** | Book: *Inference Engineering* Ch 3.3

**Prerequisite:** Day 20

## What problem does this solve?

Picking the right GPU instance for a model is a capacity planning decision with significant cost implications. Too small and the model doesn't fit. Too large and VRAM and compute sit idle. The interconnect between GPUs determines whether tensor parallelism is effective. MIG adds a third option: slice a single GPU for multi-tenant serving of smaller models.

## Concept Overview

**Multi-GPU Instance Types** vary by GPU count, total VRAM, and interconnect quality. The key dimensions: total VRAM determines which models fit; NVLink quality determines Tensor Parallelism (TP) scalability; cost determines economics.

**Multi-Instance GPU (MIG)** partitions one GPU into isolated hardware instances. Useful for:
- Serving multiple small models from one GPU
- Providing dedicated inference capacity to different tenants
- Right-sizing for models that don't need a full GPU

MIG requires enabling it in the driver and is only available on A100, H100, and newer GPUs. Performance is proportional to the number of SMs allocated.

In [1]:
!pip install -q numpy matplotlib torch 2>/dev/null
import numpy as np
import matplotlib.pyplot as plt
import torch
import time
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GB10
GPU Memory: 128.5 GB


## Part 1: Multi-GPU Instance Types

Cloud providers offer GPU instances in standardized configurations:

- **1x GPU:** Single-GPU instance. Good for small models and development.
- **4x GPU:** Mid-tier node. Can serve most 7B–70B models.
- **8x GPU:** Standard LLM node (DGX-class). Minimum for 100B+ models.
- **GB200 NVL72:** 36 Grace CPUs + 72 B200 GPUs in one rack system. Designed for disaggregated serving at scale.
- **Vera Rubin NVL72 (2026):** 36 Arm-based Vera CPUs + 72 R100 GPUs, 288 GB HBM4 per GPU. Successor to GB200.

The number of GPUs determines parallelism options and the minimum model-serving cost.

In [ ]:
# Model GPU instance configurations and their inference capabilities

instance_types = {
    "1x H100 (80GB)": {"gpus": 1, "vram_gb": 80, "nvlink": False, "cost_hr": 3.5},
    "4x H100 (320GB)": {"gpus": 4, "vram_gb": 320, "nvlink": True, "cost_hr": 14},
    "8x H100 (640GB)": {"gpus": 8, "vram_gb": 640, "nvlink": True, "cost_hr": 30},
    "8x B200 (1.5TB)": {"gpus": 8, "vram_gb": 1536, "nvlink": True, "cost_hr": 80},
    "GB200 NVL72 (13.5TB)": {"gpus": 72, "vram_gb": 13824, "nvlink": True, "cost_hr": 600},
}

model_requirements_fp16 = {
    "GPT-2 (0.1B)": 0.2,
    "LLaMA 3 8B": 16,
    "Mistral 7B": 14,
    "LLaMA 3 70B": 140,
    "DeepSeek R1 671B": 1342,
}

print("Which instance can serve which model? (weights only, FP16)")
print()
print(f"{'Model':<25} {'Size (GB)':>10}", end="")
for inst in instance_types:
    print(f" {inst[:12]:>13}", end="")
print()
print("-" * (35 + 13 * len(instance_types)))

for model, size_gb in model_requirements_fp16.items():
    print(f"{model:<25} {size_gb:>10.0f}", end="")
    for inst, specs in instance_types.items():
        # Account for KV cache (need ~20% overhead)
        fits = size_gb * 1.2 <= specs["vram_gb"]
        print(f" {'YES':>13}" if fits else f" {'NO':>13}", end="")
    print()

print()
print("Cost per token estimate (70B model, decode at 100 tokens/sec):")
for inst, specs in instance_types.items():
    if 140 * 1.2 <= specs["vram_gb"]:
        tps = 100 * min(specs["gpus"], 8)  # simplified scaling
        cost_per_million = specs["cost_hr"] / 3600 * 1e6 / tps
        print(f"  {inst}: ${cost_per_million:.2f}/M tokens")

## Part 2: NVLink and InfiniBand — GPU Interconnects

The bandwidth between GPUs determines parallelism efficiency:

- **NVLink (within node):** 900 GB/s H100, 1800 GB/s B200. Used for tensor parallelism.
- **InfiniBand (between nodes):** 400 Gb/s = ~50 GB/s. Used for pipeline parallelism.
- **PCIe (within node, no NVLink):** 64 GB/s. Limits TP effectiveness.

For tensor parallelism, the all-reduce after each transformer layer transfers `hidden_dim * batch * seq_len * 2` bytes. This must complete before the next layer can start.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model communication overhead for different interconnects and batch sizes

def allreduce_time_ms(hidden, batch_seq, bytes_per_val=2, bandwidth_gbs=900):
    """Time for all-reduce of (batch_seq, hidden) tensor."""
    bytes_total = batch_seq * hidden * bytes_per_val * 2  # *2 for ring all-reduce
    return bytes_total / (bandwidth_gbs * 1e9) * 1000

hidden = 8192  # Llama 70B
n_layers = 80
batch_sizes = [1, 4, 16, 64, 256]

interconnects = {
    "NVLink H100 (900 GB/s)": 900,
    "NVLink B200 (1800 GB/s)": 1800,
    "InfiniBand 400Gb (50 GB/s)": 50,
    "PCIe 5.0 (64 GB/s)": 64,
}

fig, ax = plt.subplots(figsize=(12, 6))

for name, bw in interconnects.items():
    overheads = [allreduce_time_ms(hidden, bs) * n_layers for bs in batch_sizes]
    ax.semilogy(batch_sizes, overheads, 'o-', linewidth=2, label=name)

ax.set_xlabel("Batch size (tokens)", fontsize=11)
ax.set_ylabel("Total all-reduce overhead (ms, log scale)", fontsize=11)
ax.set_title(f"TP Communication Overhead vs Batch Size\nhidden={hidden}, {n_layers} layers", fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print("Communication overhead vs compute time:")
compute_ms_per_layer = 1.0  # simplified: 1ms per layer for decode
total_compute = compute_ms_per_layer * n_layers

for name, bw in interconnects.items():
    overhead = allreduce_time_ms(hidden, 1) * n_layers  # batch=1 decode
    pct = overhead / total_compute * 100
    print(f"  {name}: {overhead:.2f}ms overhead ({pct:.0f}% of compute) at batch=1")

## Part 3: Multi-Instance GPU (MIG) — Slicing One GPU into Multiple

Sometimes a workload doesn't need a whole GPU. A small model (7B at FP16 = 14 GB) doesn't need 80 GB of VRAM.

NVIDIA's **Multi-Instance GPU (MIG)** partitions a single GPU into up to 7 isolated GPU instances. Each instance has:
- A dedicated slice of SMs
- A dedicated slice of HBM
- Full isolation (one instance can't access another's memory)

**MIG instances on H100 (80GB):**
- 1g.10gb: 1/7 of GPU, 10GB VRAM
- 2g.20gb: 2/7 of GPU, 20GB VRAM
- 3g.40gb: 3/7 of GPU, 40GB VRAM
- 7g.80gb: Full GPU, 80GB VRAM

**Infrastructure analogy:** MIG is like containers on a server — strong isolation between workloads, each gets a guaranteed slice of resources.

In [ ]:
# Model MIG partitioning options

h100_specs = {"total_sms": 132, "total_hbm_gb": 80, "fp16_tflops": 989}

mig_instances = {
    "1g.10gb (1/7 GPU)": {"sm_fraction": 1/7, "hbm_gb": 10},
    "2g.20gb (2/7 GPU)": {"sm_fraction": 2/7, "hbm_gb": 20},
    "3g.40gb (3/7 GPU)": {"sm_fraction": 3/7, "hbm_gb": 40},
    "7g.80gb (Full)":    {"sm_fraction": 1.0,  "hbm_gb": 80},
}

print("H100 MIG Instance Specifications")
print(f"{'Instance':<22} {'SMs':>6} {'HBM':>6} {'FP16 TFLOPS':>13} {'Max model (FP16)':>18}")
print("-" * 70)
for name, specs in mig_instances.items():
    sms = int(h100_specs["total_sms"] * specs["sm_fraction"])
    tflops = h100_specs["fp16_tflops"] * specs["sm_fraction"]
    max_model_b = specs["hbm_gb"] / 2  # FP16: 2GB per billion params
    print(f"{name:<22} {sms:>6} {specs['hbm_gb']:>5}GB {tflops:>13.0f} {max_model_b:>16.0f}B params")

print()
print("Common MIG use cases:")
print("  1g.10gb: Embedding models, small classifiers, ASR")
print("  2g.20gb: 7B LLM inference (14GB FP16), medium models")
print("  3g.40gb: 13B LLM, VLM inference")
print("  7g.80gb: 40B+ LLMs, full model serving")
print()
print("MIG vs no-MIG:")
print("  MIG: 7x the utilization for small models, full isolation, hardware guarantees")
print("  No-MIG: full bandwidth for large models, simpler setup")

## Try These Experiments

1. **Break-even analysis:** A 7B model runs on a 1xH100 (cost $3.50/hr) or a 2g.20gb MIG slice (cost ~$1/hr). If the full H100 serves 50 tokens/sec and MIG serves 30 tokens/sec, compute the traffic level at which MIG becomes cost-effective.

2. **TP vs replicas tradeoff:** For a 7B model that fits on 1 GPU, two strategies on an 8xH100 node — 8 independent replicas, or 1 replica with TP=8. At batch_size=1, which has lower latency? At batch_size=32, which has higher total throughput? (Hint: TP reduces latency; replicas increase throughput.)

3. **MIG tile configuration:** An H100 (80 GB) must serve one 40B model (80 GB FP16) and three copies of a 7B model (14 GB FP16 each). Is this possible with MIG? What partition configuration works?

## Key Takeaways

- Match the instance type to the model size — 70B needs at least 4xH100 (FP16) or 2xH100 (INT4).
- NVLink enables efficient tensor parallelism; InfiniBand is too slow for TP all-reduce between nodes.
- MIG provides hardware-isolated GPU slices — up to 7 per H100, ideal for multi-tenant serving.
- PCIe-only nodes (no NVLink) should avoid TP and use replicas instead.
- **What's next:** Day 22 — Containerization: Docker and NVIDIA NIMs.

## References
- *Inference Engineering* Ch 3.3 — Philip Kiely
- [NVIDIA Multi-Instance GPU User Guide](https://docs.nvidia.com/datacenter/tesla/mig-user-guide/)
- [NVIDIA Blackwell GB200 NVL72 architecture](https://www.nvidia.com/en-us/data-center/gb200-nvl72/)